# 03 — Limpeza Atlas Brasil (Indicadores Socioeconômicos)

**Objetivo:** processar os arquivos XLSX do Atlas Brasil, convertendo o formato wide para long e criando os datasets de indicadores socioeconômicos por estado e por município.

## Descoberta importante sobre os dados

Os arquivos XLSX do Atlas Brasil disponíveis em `datasets/atlas brasil/` contêm dados apenas para **Brasil + 27 estados**. As ~5.570 linhas de municípios listadas nos arquivos estão **completamente vazias** (verificado com openpyxl e pandas).

Isso ocorre porque o IDHM municipal do PNUD é calculado apenas para **anos censitários** (2000, 2010, possivelmente 2022). Os dados anuais municipais do Atlas Brasil são disponibilizados separadamente na plataforma online em formato diferente.

**Impacto no pipeline:**
- O join ENEM × Atlas usa `SG_UF + ano` (nível estadual) como chave principal
- `atlas_municipal.parquet` é gerado **vazio** como placeholder — quando dados municipais forem obtidos, basta populá-lo e reexecutar o notebook 04 (o join já prioriza municipal via `COALESCE`)

## Estrutura dos arquivos XLSX

Todos os arquivos têm o mesmo formato (92 colunas não-nulas):
- **Col 0:** `Territorialidades` (nome do estado/país)
- **Cols 1–13:** `INDICADOR 2012` ... `INDICADOR 2024` (valores estaduais)
- **Cols 14–26:** Desagregação BRANCO (13 anos em ordem posicional)
- **Cols 27–39:** Desagregação HOMEM
- **Cols 40–52:** Desagregação MULHER
- **Cols 53–65:** Desagregação NEGRO
- **Cols 66–78:** Desagregação RURAL ⚠️ *100% nulos — dados não publicados*
- **Cols 79–91:** Desagregação URBANO ⚠️ *100% nulos — dados não publicados*

**Inputs:** 9 arquivos XLSX em `datasets/atlas brasil/`  
**Outputs:** `data/processed/atlas/atlas_uf.parquet` + `atlas_municipal.parquet`

## 1. Imports, configuração e mapeamento de estados

In [ ]:
import pandas as pd
from pathlib import Path
from functools import reduce

Path('../data/processed/atlas').mkdir(parents=True, exist_ok=True)

ANOS_ALVO  = list(range(2012, 2025))
BASE_ATLAS = Path('../datasets/atlas brasil')

ESTADO_UF = {
    'Acre':'AC','Alagoas':'AL','Amapá':'AP','Amazonas':'AM','Bahia':'BA',
    'Ceará':'CE','Distrito Federal':'DF','Espírito Santo':'ES','Goiás':'GO',
    'Maranhão':'MA','Mato Grosso':'MT','Mato Grosso do Sul':'MS',
    'Minas Gerais':'MG','Pará':'PA','Paraíba':'PB','Paraná':'PR',
    'Pernambuco':'PE','Piauí':'PI','Rio de Janeiro':'RJ',
    'Rio Grande do Norte':'RN','Rio Grande do Sul':'RS','Rondônia':'RO',
    'Roraima':'RR','Santa Catarina':'SC','São Paulo':'SP',
    'Sergipe':'SE','Tocantins':'TO'
}


## 2. Funções de extração

### `xlsx_uf_long()`
Lê um arquivo XLSX e extrai os valores dos **indicadores principais** (colunas explicitamente nomeadas com o ano, ex: `'IDHM 2012'`) para os 27 estados, convertendo de wide para long com `pd.melt`.

O filtro `'Desag' not in str(col)` exclui as colunas de desagregação, que têm nomes repetidos e são tratadas separadamente por posição.

### `xlsx_desag_long()`
Extrai as **colunas de desagregação** (BRANCO, HOMEM, MULHER, NEGRO, RURAL, URBANO). Como essas colunas têm o mesmo nome repetido no cabeçalho (ex: `'Desagregação BRANCO IDHM PNAD'` × 13), a associação com anos é feita por **posição**: o i-ésimo elemento do grupo corresponde ao i-ésimo ano em `ANOS_ALVO`.

In [ ]:
def xlsx_uf_long(path, col_indicador):
    df = pd.read_excel(path, header=0)
    df = df[df['Territorialidades'].isin(ESTADO_UF.keys())].copy()
    df['sg_uf'] = df['Territorialidades'].map(ESTADO_UF)
    col_anos = {}
    for col in df.columns:
        for ano in ANOS_ALVO:
            if str(ano) in str(col) and 'Desag' not in str(col):
                col_anos[ano] = col; break
    value_vars = [col_anos[a] for a in ANOS_ALVO if a in col_anos]
    df_long = df[['sg_uf'] + value_vars].melt(
        id_vars='sg_uf', var_name='col_ano', value_name=col_indicador)
    df_long['ano'] = df_long['col_ano'].str.extract(r'(\d{4})').astype(int)
    return df_long[['sg_uf', 'ano', col_indicador]]

def xlsx_desag_long(path, prefixo):
    df = pd.read_excel(path, header=0)
    df = df[df['Territorialidades'].isin(ESTADO_UF.keys())].copy()
    df['sg_uf'] = df['Territorialidades'].map(ESTADO_UF)
    GRUPOS = ['BRANCO','HOMEM','MULHER','NEGRO','RURAL','URBANO']
    dfs = []
    for grupo in GRUPOS:
        desag_cols = [c for c in df.columns if 'Desag' in str(c) and grupo in str(c)]
        if not desag_cols: continue
        anos_disp = ANOS_ALVO[:len(desag_cols)]
        sub = df[['sg_uf'] + desag_cols[:len(anos_disp)]].copy()
        sub.columns = ['sg_uf'] + [str(a) for a in anos_disp]
        sub_long = sub.melt(id_vars='sg_uf', var_name='ano',
                             value_name=f'{prefixo}_{grupo.lower()}')
        sub_long['ano'] = sub_long['ano'].astype(int)
        dfs.append(sub_long)
    if not dfs: return None
    return reduce(lambda a, b: a.merge(b, on=['sg_uf','ano'], how='outer'), dfs)


## 3. Processamento de todos os arquivos

Itera sobre os 9 arquivos do Atlas Brasil, extrai os indicadores principais de cada um e exibe a cobertura (% de linhas não-nulas). Após o loop, as desagregações são extraídas do `IDHM.xlsx` separadamente.

**Arquivos processados:**
| Arquivo | Coluna gerada |
|---------|---------------|
| IDHM.xlsx | `idhm` |
| IDHM educação.xlsx | `idhm_educacao` |
| IDHM renda.xlsx | `idhm_renda` |
| IDHM longevidade.xlsx | `idhm_longevidade` |
| renda per capita.xlsx | `renda_percapita` |
| taxa de analfabetismo.xlsx | `tx_analfabetismo` |
| taxa de envelhecimento.xlsx | `tx_envelhecimento` |
| esperança de vida ao nascer.xlsx | `esperanca_vida` |
| mortalidade infantil.xlsx | `mortalidade_infantil` |

> **Nota:** `IDHM Ajustado.xlsx` (729 colunas) não foi incluído nesta versão. Contém o IDHM Ajustado à Desigualdade — pode ser adicionado em versão futura como indicador complementar para o modelo M3.

In [ ]:
ARQUIVOS = {
    'IDHM.xlsx':                'idhm',
    'IDHM educação.xlsx':        'idhm_educacao',
    'IDHM renda.xlsx':           'idhm_renda',
    'IDHM longevidade.xlsx':     'idhm_longevidade',
    'renda per capita.xlsx':     'renda_percapita',
    'taxa de analfabetismo - 15 anos ou mais de idade.xlsx': 'tx_analfabetismo',
    'taxa de envelhecimento.xlsx':'tx_envelhecimento',
    'esperança de vida ao nascer.xlsx': 'esperanca_vida',
    'mortalidade infantil.xlsx': 'mortalidade_infantil',
}

dfs_uf = []
for arquivo, col in ARQUIVOS.items():
    df_l = xlsx_uf_long(str(BASE_ATLAS / arquivo), col)
    dfs_uf.append(df_l)
    ok = df_l[col].notna().sum()
    print(f'{arquivo}: {ok}/{len(df_l)} não-nulos')

df_desag = xlsx_desag_long(str(BASE_ATLAS / 'IDHM.xlsx'), 'idhm')

atlas_uf = reduce(lambda l, r: l.merge(r, on=['sg_uf','ano'], how='outer'), dfs_uf)
if df_desag is not None:
    atlas_uf = atlas_uf.merge(df_desag, on=['sg_uf','ano'], how='left')
atlas_uf = atlas_uf[atlas_uf['ano'].isin(ANOS_ALVO)].copy()
print(f'\natlas_uf: {atlas_uf.shape}')
atlas_uf.head(3)


## 4. Merge final e exportação

Todos os DataFrames long são mesclados pela chave `['sg_uf', 'ano']` com `outer join`, garantindo que nenhum estado/ano seja perdido se faltar em algum arquivo.

**atlas_uf.parquet:** 351 linhas (27 UFs × 13 anos), 100% de cobertura nos indicadores principais.

**atlas_municipal.parquet:** gerado com o mesmo schema mas **sem linhas** — serve como placeholder para dados municipais futuros. Quando populado, o notebook 04 priorizará automaticamente o dado municipal via `COALESCE` sem necessidade de alteração de código.

> ⚠️ `idhm_rural` e `idhm_urbano` estarão 100% nulos — os dados de desagregação rural/urbano não estão disponíveis no Atlas Brasil para o período analisado. Essas colunas devem ser **excluídas** do modelo M3.

In [ ]:
# Salva atlas estadual
atlas_uf.to_parquet('../data/processed/atlas/atlas_uf.parquet', index=False)
print('Salvo: atlas_uf.parquet')

# Cria atlas municipal vazio com o mesmo schema (preenchido quando tiver dados)
INDICADORES = [c for c in atlas_uf.columns if c not in ['sg_uf','ano']]
atlas_muni = pd.DataFrame(columns=['code_muni','ano'] + INDICADORES)
atlas_muni.to_parquet('../data/processed/atlas/atlas_municipal.parquet', index=False)
print('Salvo: atlas_municipal.parquet (vazio — placeholder para dados municipais futuros)')

# Cobertura
print('\nCobertura atlas_uf:')
for col in INDICADORES:
    print(f'  {col}: {100*atlas_uf[col].notna().mean():.1f}%')
